# Variance Reduction in Active Learning - Starter Notebook
This notebook uses a bootstrap committee to estimate predictive variance and queries points with the highest variance.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

In [ ]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=7, random_state=42)
X_labeled, y_labeled = X[:140], y[:140]
X_unlabeled = X[140:700]

rng = np.random.default_rng(42)
committee_probs = []
for _ in range(12):
    bootstrap_idx = rng.choice(len(X_labeled), size=len(X_labeled), replace=True)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_labeled[bootstrap_idx], y_labeled[bootstrap_idx])
    committee_probs.append(model.predict_proba(X_unlabeled)[:, 1])

committee_probs = np.vstack(committee_probs)
variance_scores = committee_probs.var(axis=0)
query_ids = np.argsort(variance_scores)[-25:]

In [ ]:
pd.DataFrame({
    'unlabeled_index': query_ids,
    'predictive_variance': variance_scores[query_ids],
}).sort_values('predictive_variance', ascending=False).head(10).round(5)